In [1]:
# 4a_feature_eng_ukhls.ipynb


# Applies feature engineering to the backfilled UKHLS Wave O data.
# All operations are driven by config_variables.py — no hardcoded lists.
# Original columns are NEVER overwritten — all modifications create a new
# column suffixed with _eng (e.g. o_doby_dv → o_doby_dv_eng).
# OHE output columns also carry the _eng infix: o_racel_dv_eng_1 etc.
# Steps:
#   1. Load o_indresp_backfilled.pkl (UKHLS negatives already → -1 from 3a)
#   2. Apply value recodes (RECODE_MAPS)     → col_eng
#   3. Apply floor clipping  (FLOOR_VALUES)  → col_eng
#   4. Apply upper clipping  (CLIP_VALUES)   → col_eng
#   5. One-hot encode categorical variables  (ONE_HOT_VARS) → col_eng_N
#   6. Force ALL columns to float32 (includes OHE columns added in step 5)
#   7. Save as o_indresp_feature_eng.pkl


import sys, os
sys.path.insert(0, os.path.abspath('..'))


import importlib
from data_pipeline.config_paths import USE_TEST_DATA, DATA_FOLDER
import data_pipeline.config_variables as _ukhls_vars
_ukhls_vars.reload_config_variables()


import pandas as pd
import numpy as np
from data_pipeline.helpers.pipeline_checks import warn_non_numeric


from data_pipeline.config_variables import (
    RECODE_MAPS, FLOOR_VALUES, CLIP_VALUES, ONE_HOT_VARS, VARIABLES, TRANSFORMS,
 )


# ── Config ────────────────────────────────────────────────────────────────────
WAVE       = "o"
INPUT_PKL  = f"../{DATA_FOLDER}/3_backfill_ukhls_waves/o_indresp_backfilled.pkl"
OUTPUT_PKL = f"../{DATA_FOLDER}/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"


def get_base_code(col_name, wave_prefix):
    prefix = f"{wave_prefix}_"
    return col_name[len(prefix):] if col_name.startswith(prefix) else col_name


# ── 1. Load ───────────────────────────────────────────────────────────────────
print(f"Loading {INPUT_PKL} ...")
df = pd.read_pickle(INPUT_PKL)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")


# ── 1b. Apply transforms (e.g. birth year → age) ──────────────────────────
if TRANSFORMS:
    print("\nStep 1b: Applying transforms ...")
    for base, tfm in TRANSFORMS.items():
        col = f"{WAVE}_{base}"
        col_eng = f"{col}_eng"
        if col not in df.columns:
            print(f"  SKIP {col} — not in dataframe")
            continue
        if tfm == "birth_year_to_age":
            current_year = pd.Timestamp.now().year
            raw = pd.to_numeric(df[col], errors='coerce')
            df[col_eng] = current_year - raw
            df.loc[df[col_eng] < 0, col_eng] = np.nan
            print(f"  {col} → {col_eng}: birth_year_to_age (current_year={current_year})")
        else:
            print(f"  WARNING: unknown transform '{tfm}' for {col} — skipping")


# ── 2. Value Recodes ──────────────────────────────────────────────────────────
# Writes to col_eng. If col_eng already exists (from a transform in step 1b),
# the recode is applied on top of the transformed values.
print("\nStep 2: Applying value recodes ...")
recoded = []
for col in list(df.columns):
    if col.endswith("_eng"):
        continue  # skip already-engineered columns
    base = get_base_code(col, WAVE)
    recode = RECODE_MAPS.get(base)
    var_def = VARIABLES.get(base, {})
    categories = set((var_def.get('categories') or {}).keys())
    if recode:
        col_eng = f"{col}_eng"
        # If _eng already exists (e.g. from a transform), recode on top of it
        src_col = col_eng if col_eng in df.columns else col
        col_numeric = pd.to_numeric(df[src_col], errors='coerce')
        unique_vals = set(col_numeric.dropna().unique())
        mapped_vals = set(recode.keys())
        allowed_vals = categories | mapped_vals
        not_listed = unique_vals - allowed_vals
        if not_listed:
            not_listed_sorted = sorted(not_listed)
            not_listed_str = ', '.join(str(x) for x in not_listed_sorted)
            print(f"WARNING: {col} has values not in categories or recode map: {not_listed_str}")
        df[col_eng] = col_numeric.replace(recode)
        recoded.append(f"  {col} → {col_eng}: {recode}")
if recoded:
    print("\n".join(recoded))
else:
    print("  (none)")


# ── 3. Floor Clipping ─────────────────────────────────────────────────────────
print("\nStep 3: Applying floor values ...")
for col in list(df.columns):
    if col.endswith("_eng"):
        continue
    base = get_base_code(col, WAVE)
    floor_val = FLOOR_VALUES.get(base)
    if floor_val is not None:
        col_eng = f"{col}_eng"
        src_col = col_eng if col_eng in df.columns else col
        numeric = pd.to_numeric(df[src_col], errors='coerce')
        not_provided = numeric == -1.0
        floored = numeric.clip(lower=floor_val)
        floored = floored.where(~not_provided, -1.0)
        n_floored = int((floored != numeric).sum())
        df[col_eng] = floored
        print(f"  {src_col} → {col_eng}: floored at {floor_val}  ({n_floored:,} rows affected; -1.0 left as not provided)")


# ── 4. Upper Clipping ─────────────────────────────────────────────────────────
print("\nStep 4: Applying clip values ...")
for col in list(df.columns):
    if col.endswith("_eng"):
        continue
    base = get_base_code(col, WAVE)
    clip_val = CLIP_VALUES.get(base)
    if clip_val is not None:
        col_eng = f"{col}_eng"
        src_col = col_eng if col_eng in df.columns else col
        numeric = pd.to_numeric(df[src_col], errors='coerce')
        n_clipped = int((numeric > clip_val).sum())
        df[col_eng] = numeric.clip(upper=clip_val)
        print(f"  {src_col} → {col_eng}: clipped at {clip_val:,}  ({n_clipped:,} rows affected)")


# ── 4b. Derive binary features from config ─────────────────────────────────────
# Dynamically derive binary features as specified in config_variables.py.
# Reads from the _eng column if it exists (variable has been recoded/transformed).
print("\nStep 4b: Deriving binary features from config ...")
for base, var_def in VARIABLES.items():
    derrived_cfg = var_def.get('create_binary_derrived_feature')
    if derrived_cfg:
        col = f"{WAVE}_{base}"
        col_eng = f"{col}_eng"
        src_col = col_eng if col_eng in df.columns else col
        derrived_col = f"{WAVE}_{derrived_cfg['feature_name']}"
        threshold = derrived_cfg.get('threshold', 0)
        if src_col in df.columns:
            df[derrived_col] = (pd.to_numeric(df[src_col], errors='coerce') > threshold).astype(np.float32)
            n_with = int((df[derrived_col] == 1.0).sum())
            print(f"  {derrived_col}: {n_with:,} rows > {threshold} ({100*n_with/len(df):.1f}%)")
        else:
            print(f"  SKIP — {src_col} not in dataframe")


# ── 5. One-hot encoding ───────────────────────────────────────────────────────
# For each variable with one_hot defined, create binary indicator columns.
# Source: uses col_eng if it exists (variable was recoded/transformed), else col.
# Naming: {wave}_{base}_eng_{int(code)}  e.g. o_jbstat_eng_1 = 1 if employed
#         {wave}_{base}_eng_not_answered for any value not in the OHE code list.
# Type casting to float32 is handled centrally in step 6.
print("\nStep 5: One-hot encoding ...")
oh_new_cols = []
for base, one_hot_spec in ONE_HOT_VARS.items():
    col = f"{WAVE}_{base}"
    col_eng = f"{col}_eng"
    # Use _eng source if available (variable has been recoded/transformed)
    src_col = col_eng if col_eng in df.columns else col
    if src_col not in df.columns:
        print(f"  SKIP {col} — not in dataframe")
        continue

    src = pd.to_numeric(df[src_col], errors='coerce')

    # Determine which codes to encode
    if one_hot_spec is True:
        # Prefer group_labels keys (post-recode canonical codes) when present;
        # fall back to categories keys so we don't create zero-filled columns
        # for raw codes that have been recoded away.
        var_def = VARIABLES[base]
        codes = list((var_def.get("group_labels") or var_def["categories"]).keys())
    else:
        # one_hot_spec is a list of specific codes
        codes = list(one_hot_spec)

    codes_ohe = {float(c) for c in codes}
    not_answered = src.notna() & ~src.isin(codes_ohe)

    for code in codes:
        new_col = f"{WAVE}_{base}_eng_{int(code)}"
        df[new_col] = (src == code) & (~not_answered)
        oh_new_cols.append(new_col)
        print(f"  {new_col}  ({int((df[new_col]).sum()):,} = 1)")

    na_col = f"{WAVE}_{base}_eng_not_answered"
    df[na_col] = not_answered
    oh_new_cols.append(na_col)
    print(f"  {na_col}  ({int(not_answered.sum()):,} = 1)")


print(f"\n  {len(oh_new_cols)} binary indicator columns added; originals kept.")


# ── 6. Force ALL columns to float32 ──────────────────────────────────────────
# Runs after step 5, so OHE columns added above are included.
print("\nStep 6: Converting all features to float32 ...")
for col in df.columns:
    if col == 'pidp':
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.int64)
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float32)


n_nulls = df.drop(columns=['pidp']).isna().sum().sum()
if n_nulls > 0:
    print(f"  WARNING — {n_nulls:,} NaN values remain after feature engineering")
    print(df.drop(columns=['pidp']).isna().sum()[lambda s: s > 0])
else:
    print("  No NaN values — clean feature matrix")
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float32)
warn_non_numeric(df, "4a_feature_eng_ukhls")


# ── 7. Save ───────────────────────────────────────────────────────────────────
df.to_pickle(OUTPUT_PKL, protocol=5)
print(f"\nDone. Feature matrix saved to {OUTPUT_PKL}")
print(f"Shape: {df.shape}")
print(df.dtypes)
print(df.dtypes)




🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Loading ../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl ...
Loaded 19,618 rows × 11 columns

Step 1b: Applying transforms ...
  o_doby_dv: birth_year_to_age (current_year=2026)

Step 2: Applying value recodes ...
  o_jbstat: {1.0: 1.0, 2.0: 1.0, 12.0: 1.0, 13.0: 1.0, 3.0: 3.0, 4.0: 4.0, 5.0: 5.0, 6.0: 5.0, 14.0: 5.0, 15.0: 5.0, 7.0: 7.0, 9.0: 7.0, 11.0: 7.0, 8.0: 8.0, 10.0: 8.0, 97.0: 8.0}
  o_hiqual_dv: {9.0: 6.0}
  o_hhtype_dv: {1.0: 1.0, 2.0: 1.0, 3.0: 1.0, 4.0: 2.0, 5.0: 3.0, 6.0: 4.0, 8.0: 4.0, 10.0: 5.0, 11.0: 6.0, 12.0: 7.0, 16.0: 8.0, 17.0: 8.0, 18.0: 9.0, 19.0: 10.0, 20.0: 11.0, 21.0: 12.0, 22.0: 13.0, 23.0: 14.0}
  o_tenure_dv: {1.0: 1.0, 2.0: 1.0, 3.0: 2.0, 4.0: 2.0,